In [ ]:
# %% [markdown]
# # M5 Decagon Ensemble: Hierarchical WRMSSE Audit
# **Aim:** Deconstruct the WRMSSE across all 12 levels of the Walmart hierarchy.
# **Research Focus:** Error Attribution, Bias Detection, and Aggregation Consistency.

# %%
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.metrics import WRMSSEMetric, HierarchicalAggregator

# %% [markdown]
# ### 1. The Mathematical Projection
# We use the Sparse Matrix $S \in \{0, 1\}^{42840 \times 30490}$ to aggregate 
# bottom-level forecasts to all 12 hierarchical levels.
# $$ \hat{\mathbf{Y}}_{all} = S \hat{\mathbf{Y}}_{item} $$

# %%
# 1. Load Global Predictions & Ground Truth
# Assuming you've run scripts/predict.py and saved the raw tensor
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

preds = torch.load("../outputs/preds/final_forecast.pt", map_location='cpu') # [30490, 28]
actuals = torch.load("../data/processed/targets_train.pt", map_location='cpu')
weights = torch.load("../data/processed/m5_weights.pt", map_location='cpu')
scales = torch.load("../data/processed/m5_scales.pt", map_location='cpu')
S_matrix = torch.load("../data/processed/m5_agg_matrix_sparse.pt", map_location='cpu')

aggregator = HierarchicalAggregator(S_matrix)

# %%
# 2. Perform Vectorized Aggregation
# Projected shapes: [42840, 28]
print("Projecting forecasts through the hierarchy...")
y_hat_all = aggregator.aggregate(preds)    
y_true_all = aggregator.aggregate(actuals) 

# %% [markdown]
# ### 2. Level-wise Error Breakdown
# We analyze the WRMSSE per level. 
# Level 1: Total | Level 2: State | ... | Level 12: Item-Store

# %%
level_map = {
    "Level 1: Total": (0, 1),
    "Level 2: State": (1, 4),
    "Level 3: Store": (4, 14),
    "Level 4: Category": (14, 17),
    "Level 9: Unit-State": (1222, 1231),
    "Level 12: Item-Store": (12350, 42840)
}

metric_engine = WRMSSEMetric(weights, scales, device=torch.device('cpu'))

results = []
for level, (start, end) in level_map.items():
    l_preds = y_hat_all[start:end]
    l_true = y_true_all[start:end]
    l_weights = weights[start:end]
    l_scales = scales[start:end]
    
    # Calculate RMSE and RMSSE for this specific level
    mse = torch.mean((l_preds - l_true)**2, dim=1)
    rmsse = torch.sqrt(mse / (l_scales + 1e-10))
    wrmsse = torch.sum(l_weights * rmsse).item()
    
    results.append({"Level": level, "WRMSSE": wrmsse})

df_results = pd.DataFrame(results)
print(df_results)

# %% [markdown]
# ### 3. Visualizing Bias (Total Level)
# Level 1 is the most 'visible' error. If your model is consistently 
# over-predicting the total sales, it indicates a Global Bias.

# %%
total_pred = y_hat_all[0].numpy()
total_true = y_true_all[0].numpy()

plt.figure(figsize=(14, 6))
plt.plot(total_true, label="Actual Total Sales", color='black', lw=2)
plt.plot(total_pred, label="Decagon Ensemble Forecast", color='crimson', ls='--')
plt.fill_between(range(28), total_true, total_pred, color='crimson', alpha=0.1)
plt.title("Level 1 Audit: Total Walmart Sales Forecast (28 Days)")
plt.xlabel("Day in Horizon")
plt.ylabel("Units")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# %% [markdown]
# ### 4. Expert Trust Correlation
# We correlate the Meta-Blender's Trust Scores with the error. 
# If 'VAT-GNN' weight was high during a spike, did the error stay low?

# %%
trust_scores = torch.load("../outputs/logs/inference_trust_analysis.pt", map_location='cpu') # [30490, 9]
avg_trust = trust_scores.mean(dim=0).numpy()
expert_names = ["H-GNN", "C-GNN", "Graphormer", "SigGNN", "ZI-GNN", "E-GNN", "CalGNN", "FlowGNN", "VAT-GNN"]

plt.figure(figsize=(10, 6))
sns.barplot(x=expert_names, y=avg_trust, palette="magma")
plt.xticks(rotation=45)
plt.title("Global Expert Trust Allocation (Inference Phase)")
plt.ylabel("Mean Attention Weight")
plt.show()